# GAN-Augmented Heart Sound Classification

This notebook explains the full project in beginner-friendly steps. The goal is to classify heart sound WAV files as **Normal** or **Abnormal** using MFCC + log-mel features, a small GAN for class balancing, and a compact CNN + BiGRU classifier.


## 1. Big Picture

The pipeline is:

```text
WAV audio -> filtering -> fixed-length segments -> MFCC + log-mel features -> GAN augmentation -> CNN + BiGRU classifier
```

Why use a GAN? Medical datasets often have fewer abnormal examples. The GAN learns from abnormal feature matrices and creates extra abnormal-like examples so the classifier sees a more balanced training set.


## 2. Imports

These imports are enough to inspect the data, models, and saved outputs.


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

from preprocessing import preprocess_signal, segment_signal, extract_features
from gan_model import build_generator, build_discriminator
from cnn_classifier import build_cnn_classifier


## 3. Preprocessing

`preprocessing.py` does four important things:

1. Reads WAV files.
2. Resamples audio to 2000 Hz when needed.
3. Applies a 25-400 Hz bandpass filter and normalization.
4. Splits audio into 2.52 second windows and extracts `(64, 71)` features.

Run this from a terminal before training:

```bash
python preprocessing.py
```


In [ ]:
processed_dir = "data/processed"

if os.path.exists(os.path.join(processed_dir, "X_train.npy")):
    X_train = np.load(os.path.join(processed_dir, "X_train.npy"))
    y_train = np.load(os.path.join(processed_dir, "y_train.npy"))
    print("X_train shape:", X_train.shape)
    print("y_train shape:", y_train.shape)
    print("Normal samples:", np.sum(y_train == 0))
    print("Abnormal samples:", np.sum(y_train == 1))
else:
    print("Processed arrays not found yet. Run: python preprocessing.py")


## 4. Feature Shape

Each training example is a matrix with shape `(64, 71)`:

- `64` = time frames.
- `71` = 39 MFCC-style features + 32 log-mel spectrogram features.

A model can read this like a small image: time on one axis, audio features on the other.


In [ ]:
if "X_train" in globals() and len(X_train) > 0:
    plt.figure(figsize=(7, 3))
    plt.imshow(X_train[0].T, aspect="auto", origin="lower", cmap="coolwarm")
    plt.title("One MFCC feature matrix")
    plt.xlabel("Time frame")
    plt.ylabel("Feature index")
    plt.colorbar()
    plt.show()


## 5. GAN Model

The generator converts random noise into a fake abnormal feature matrix. The discriminator tries to tell whether a matrix is real or fake.

Run training from a terminal:

```bash
python train_gan.py --epochs 50 --batch_size 32
```


In [ ]:
generator = build_generator(latent_dim=100, feature_dim=71)
discriminator = build_discriminator(input_shape=(64, 71))

fake_batch = generator(np.random.normal(size=(2, 100)).astype("float32"))
print("Generator output shape:", fake_batch.shape)
print("Discriminator output shape:", discriminator(fake_batch).shape)


## 6. CNN Classifier

The classifier is intentionally small:

- Conv1D learns local time patterns.
- MaxPooling compresses the time axis.
- GlobalAveragePooling turns the sequence into one vector.
- Dense layers output one probability.

Run training from a terminal:

```bash
python train_classifier.py --epochs 25 --batch_size 16
```


In [ ]:
classifier = build_cnn_classifier(input_shape=(64, 71))
classifier.summary()


## 7. Evaluation

After training, compare the baseline and GAN-augmented models:

```bash
python evaluation.py
```

Important metrics:

- Accuracy: overall correctness.
- Precision: when the model says abnormal, how often it is right.
- Recall: how many abnormal cases it catches.
- AUC: how well probabilities rank normal vs abnormal samples.


In [ ]:
results_csv = "outputs/gan_vs_no_gan_results.csv"
if os.path.exists(results_csv):
    import pandas as pd
    display(pd.read_csv(results_csv))
else:
    print("Results not found yet. Run: python evaluation.py")


## 8. Single WAV Prediction

After training the classifier, test one WAV file:

```bash
python predict.py --wav data/raw/a0007.wav
```

The model predicts each segment, then averages segment probabilities to produce one final recording-level result.


## 9. Beginner Study Checklist

To understand this project deeply, study in this order:

1. `preprocessing.py`: how raw audio becomes numbers.
2. `cnn_classifier.py`: how the classifier reads feature sequences.
3. `gan_model.py`: how generator and discriminator are built.
4. `train_gan.py`: how adversarial training works.
5. `train_classifier.py`: how GAN samples balance the dataset.
6. `evaluation.py`: how to decide whether the model really improved.
